In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [2]:
SEED = 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f"Artifacts will be saved to: {ARTIFACTS_DIR}/")

Using device: cpu
Artifacts will be saved to: artifacts/


In [3]:
# Загружаем датасет emotion
# Классификация текста по 6 эмоциям:
# sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5)
dataset = load_dataset("emotion")
print(dataset)

# Названия классов
label_names = dataset["train"].features["label"].names
print(f"\nКлассы ({len(label_names)}): {label_names}")

# Размеры split-ов
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} примеров")

print("\n--- 5 примеров из train ---")
for i in range(5):
    example = dataset["train"][i]
    print(f"  [{label_names[example['label']]}] {example['text']}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Классы (6): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
  train: 16000 примеров
  validation: 2000 примеров
  test: 2000 примеров

--- 5 примеров из train ---
  [sadness] i didnt feel humiliated
  [sadness] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
  [anger] im grabbing a minute to post i feel greedy wrong
  [love] i am ever feeling nostalgic about the fireplace i will know that it is still on the property
  [anger] i am feeling grouchy


In [4]:
train_dataset = dataset["train"]       # 16000 
val_dataset   = dataset["validation"]  # 2000 
test_dataset  = dataset["test"]        # 2000 

print(f"Train size:      {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size:       {len(test_dataset)}")

NUM_LABELS = len(label_names)
print(f"Число классов:   {NUM_LABELS}")

Train size:      16000
Validation size: 2000
Test size:       2000
Число классов:   6


In [5]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


sample_texts = [dataset["train"][i]["text"] for i in range(5)]

print("+" * 60)
print("ДЕМОНСТРАЦИЯ ТОКЕНИЗАЦИИ")
print("+" * 60)

for i, text in enumerate(sample_texts):
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
    
    print(f"\n[Пример {i+1}]")
    print(f"  Текст:          {text[:80]}...")
    print(f"  Токены:         {tokens[:12]}...")
    print(f"  input_ids:      {encoding['input_ids'][0][:12].tolist()}...")
    print(f"  attention_mask: {encoding['attention_mask'][0][:12].tolist()}...")

# --- Демонстрация special tokens ---
print("\nSpecial tokens")
print(f"  [CLS] id: {tokenizer.cls_token_id}  -> токен: {tokenizer.cls_token}")
print(f"  [SEP] id: {tokenizer.sep_token_id}  -> токен: {tokenizer.sep_token}")
print(f"  [PAD] id: {tokenizer.pad_token_id}  -> токен: {tokenizer.pad_token}")

# --- Демонстрация padding и truncation ---
print("\nПример padding (короткий текст, max_length=20)")
short_text = "I am happy"
enc_short = tokenizer(short_text, padding="max_length", max_length=20, return_tensors="pt")
print(f"  Текст:          '{short_text}'")
print(f"  input_ids:      {enc_short['input_ids'][0].tolist()}")
print(f"  attention_mask: {enc_short['attention_mask'][0].tolist()}")

print("\nПример truncation (длинный текст, max_length=10)")
long_text = "I feel so incredibly sad and disappointed because everything went wrong today"
enc_long = tokenizer(long_text, truncation=True, max_length=10, return_tensors="pt")
tokens_long = tokenizer.convert_ids_to_tokens(enc_long["input_ids"][0])
print(f"  Текст:    '{long_text}'")
print(f"  Токены:   {tokens_long}")
print(f"  (оригинал был длиннее, обрезан до 10 токенов включая [CLS] и [SEP])")

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
ДЕМОНСТРАЦИЯ ТОКЕНИЗАЦИИ
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

[Пример 1]
  Текст:          i didnt feel humiliated...
  Токены:         ['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']...
  input_ids:      [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0]...
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]...

[Пример 2]
  Текст:          i can go from feeling so hopeless to so damned hopeful just from being around so...
  Токены:         ['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful']...
  input_ids:      [101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772]...
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...

[Пример 3]
  Текст:          im grabbing a minute to post i feel greedy wrong...
  Токены:         ['[CLS]', 'im', 'grabbing', 'a', 'minute', 'to'

In [6]:
# Предобученка

print("+" * 60)
print("ИНФЕРЕНС ГОТОВОЙ PRETRAINED МОДЕЛИ")
print("+" * 60)

# Модель обучена на sentiment (pos/neg/neu) — не на наших 6 классах
sentiment_pipeline = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if DEVICE == "cuda" else -1,
)

inference_examples = [
    {"text": "I am so happy today, everything is wonderful!", "true_label": "joy"},
    {"text": "I feel so sad and lonely, nobody cares.",        "true_label": "sadness"},
    {"text": "I am furious, this is absolutely unacceptable!", "true_label": "anger"},
    {"text": "I love you so much, you mean the world to me.", "true_label": "love"},
    {"text": "I am scared, something is very wrong.",          "true_label": "fear"},
]

print(f"\nМодель: distilbert-base-uncased-finetuned-sst-2-english")
print(f"(обучена на SST-2: binary sentiment — POSITIVE / NEGATIVE)\n")

for ex in inference_examples:
    result = sentiment_pipeline(ex["text"])[0]
    print(f"  Текст:      {ex['text']}")
    print(f"  True label: {ex['true_label']}")
    print(f"  Предсказание модели: {result['label']} (score={result['score']:.3f})")
    print()

print("+++Вывод+++")
print(
    "Готовая SST-2 модель умеет различать позитивные и негативные тексты,\n"
    "но не может корректно различить joy/love/surprise (всё маппится в POSITIVE)\n"
    "и sadness/anger/fear (всё в NEGATIVE). Такая модель не подходит для нашей\n"
    "6-классовой задачи напрямую — нужен fine-tuning на нашем датасете."
)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
ИНФЕРЕНС ГОТОВОЙ PRETRAINED МОДЕЛИ
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Модель: distilbert-base-uncased-finetuned-sst-2-english
(обучена на SST-2: binary sentiment — POSITIVE / NEGATIVE)

  Текст:      I am so happy today, everything is wonderful!
  True label: joy
  Предсказание модели: POSITIVE (score=1.000)

  Текст:      I feel so sad and lonely, nobody cares.
  True label: sadness
  Предсказание модели: NEGATIVE (score=1.000)

  Текст:      I am furious, this is absolutely unacceptable!
  True label: anger
  Предсказание модели: NEGATIVE (score=1.000)

  Текст:      I love you so much, you mean the world to me.
  True label: love
  Предсказание модели: POSITIVE (score=1.000)

  Текст:      I am scared, something is very wrong.
  True label: fear
  Предсказание модели: NEGATIVE (score=0.999)

+++Вывод+++
Готовая SST-2 модель умеет различать позитивные и негативные тексты,
но не может корректно различить joy/love/surprise (всё маппится в POSITIVE)
и sadness/anger/fear (всё в NEGATIVE). Такая модель не подходит для нашей
6-классовой задачи напрямую — ну

In [7]:
MAX_LENGTH = 64

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

print("Токенизация train/val/test...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val   = val_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

# Устанавливаем формат для PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_val.set_format("torch",   columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch",  columns=["input_ids", "attention_mask", "label"])

print("Токенизация завершена.")
print(f"  Train: {len(tokenized_train)} примеров")
print(f"  Val:   {len(tokenized_val)} примеров")
print(f"  Test:  {len(tokenized_test)} примеров")

Токенизация train/val/test...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Токенизация завершена.
  Train: 16000 примеров
  Val:   2000 примеров
  Test:  2000 примеров


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)
model.to(DEVICE)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",         
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=100,
    report_to="none",
)

print(f"TrainingArguments создан успешно)")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Запуск fine-tuning...")
train_result = trainer.train()
print("Fine-tuning завершён.")
print(f"  Training loss: {train_result.training_loss:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Kirill\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainingArguments создан успешно)
Запуск fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.244879,0.232766,0.924000,0.900520
2,0.159641,0.174322,0.930500,0.904211
3,0.109668,0.165020,0.934500,0.911057


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Kirill\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Fine-tuning завершён.
  Training loss: 0.2974


In [20]:
print("Оценка лучшей модели на VALIDATION:")

val_output = trainer.predict(tokenized_val) 
val_logits = val_output.predictions
val_labels = val_output.label_ids
val_preds  = np.argmax(val_logits, axis=-1)

val_acc = accuracy_score(val_labels, val_preds)
val_f1  = f1_score(val_labels, val_preds, average="macro")

print(f"  Accuracy:  {val_acc:.4f}")
print(f"  F1 macro:  {val_f1:.4f}")

Оценка лучшей модели на VALIDATION:


C:\Users\Kirill\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Accuracy:  0.9345
  F1 macro:  0.9111


In [22]:
print("+" * 50)
print("ФИНАЛЬНАЯ ОЦЕНКА НА TEST SET")
print("+" * 50)

test_output = trainer.predict(tokenized_test)
logits      = test_output.predictions
labels      = test_output.label_ids
pred_ids    = np.argmax(logits, axis=-1)
confidence  = torch.softmax(
    torch.tensor(logits), dim=-1
).numpy().max(axis=-1)

test_acc = accuracy_score(labels, pred_ids)
test_f1  = f1_score(labels, pred_ids, average="macro")

print(f"  Accuracy:  {test_acc:.4f}")
print(f"  F1 macro:  {test_f1:.4f}")

# Для дальнейших ячеек (матрица ошибок, анализ ошибок)
true_label_names = [label_names[i] for i in labels]
pred_label_names = [label_names[i] for i in pred_ids]

++++++++++++++++++++++++++++++++++++++++++++++++++
ФИНАЛЬНАЯ ОЦЕНКА НА TEST SET
++++++++++++++++++++++++++++++++++++++++++++++++++


C:\Users\Kirill\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Accuracy:  0.9275
  F1 macro:  0.8830


In [24]:
# Получаем предсказания на test
predictions_output = trainer.predict(tokenized_test)
logits    = predictions_output.predictions
labels    = predictions_output.label_ids
pred_ids  = np.argmax(logits, axis=-1)
confidence = torch.softmax(torch.tensor(logits), dim=-1).numpy().max(axis=-1)

# Метрики
acc    = accuracy_score(labels, pred_ids)
f1_mac = f1_score(labels, pred_ids, average="macro")
print(f"Test Accuracy:  {acc:.4f}")
print(f"Test F1 macro:  {f1_mac:.4f}")

# True/pred labels (строки)
true_label_names = [label_names[i] for i in labels]
pred_label_names = [label_names[i] for i in pred_ids]

C:\Users\Kirill\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Accuracy:  0.9275
Test F1 macro:  0.8830


In [25]:
cm = confusion_matrix(labels, pred_ids)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Confusion Matrix — Test Set\n(distilbert-base-uncased, fine-tuned on emotion)")
plt.tight_layout()

cm_path = os.path.join(ARTIFACTS_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Матрица ошибок сохранена: {cm_path}")

Матрица ошибок сохранена: artifacts\confusion_matrix.png


C:\Users\Kirill\AppData\Local\Temp\ipykernel_14380\1800942205.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# Берём все тексты из test (оригинальные, до токенизации)
test_texts = test_dataset["text"]

# Собираем DataFrame
predictions_df = pd.DataFrame({
    "text":       test_texts,
    "true_label": true_label_names,
    "pred_label": pred_label_names,
    "confidence": confidence.round(4),
})

csv_path = os.path.join(ARTIFACTS_DIR, "sample_predictions.csv")
predictions_df.to_csv(csv_path, index=False)
print(f"sample_predictions.csv сохранён: {csv_path}")
print(f"Всего строк: {len(predictions_df)}")
print(predictions_df.head(10).to_string(index=False))

sample_predictions.csv сохранён: artifacts\sample_predictions.csv
Всего строк: 2000
                                                                                                                                                                                                                          text true_label pred_label  confidence
                                                                                                                                                                   im feeling rather rotten so im not very ambitious right now    sadness    sadness      0.9960
                                                                                                                                                                                     im updating my blog because i feel shitty    sadness    sadness      0.9970
                                                                                                                             i never make her sep

In [27]:
print("+" * 60)
print("АНАЛИЗ ОШИБОК МОДЕЛИ")
print("+" * 60)

# Найдём ошибочные предсказания
errors_df = predictions_df[predictions_df["true_label"] != predictions_df["pred_label"]].copy()
errors_df = errors_df.sort_values("confidence", ascending=False)

total     = len(predictions_df)
n_errors  = len(errors_df)
print(f"Всего примеров в test:    {total}")
print(f"Ошибочных предсказаний:   {n_errors} ({n_errors/total*100:.1f}%)")

print(f"\n+++ 10 примеров ошибок (с высокой уверенностью модели) +++")
for _, row in errors_df.head(10).iterrows():
    print(f"  True: {row['true_label']:10s} | Pred: {row['pred_label']:10s} "
          f"| Conf: {row['confidence']:.3f} | Text: {row['text'][:70]}")

# Анализ пар (true, pred)
print(f"\n+++ Наиболее частые пары ошибок (true -> pred) +++")
error_pairs = (
    errors_df.groupby(["true_label", "pred_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(10)
)
print(error_pairs.to_string(index=False))

print(
    "\n+++ Комментарий +++\n"
    "Наиболее распространённые ошибки:\n"
    "  - fear <-> sadness: эмоции лексически близки, тексты содержат похожие слова\n"
    "    ('scared', 'worried' vs 'lonely', 'hopeless').\n"
    "  - anger <-> sadness: разочарование может выражаться словами обеих эмоций.\n"
    "  - love <-> joy: положительные эмоции часто смешиваются.\n"
    "Ошибки с высокой уверенностью указывают на то, что граница между\n"
    "смежными эмоциями в тексте действительно размыта."
)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
АНАЛИЗ ОШИБОК МОДЕЛИ
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Всего примеров в test:    2000
Ошибочных предсказаний:   145 (7.2%)

+++ 10 примеров ошибок (с высокой уверенностью модели) +++
  True: surprise   | Pred: sadness    | Conf: 0.996 | Text: i cannot even begin to express in words the depth of sorrow that i fee
  True: anger      | Pred: joy        | Conf: 0.996 | Text: whenever i put myself in others shoes and try to make the person happy
  True: anger      | Pred: sadness    | Conf: 0.995 | Text: i actually was in a meeting last week where someone yelled at an older
  True: surprise   | Pred: sadness    | Conf: 0.993 | Text: i just feel are ludicrous and wasting space or so trite they should ha
  True: fear       | Pred: sadness    | Conf: 0.993 | Text: i feel inside cause life is like a game sometimes but then you came ar
  True: sadness    | Pred: fear       | Conf: 0.992 | Text: i feel unprote

Наиболее распространённые ошибки:
- fear <-> sadness: эмоции лексически близки, тексты содержат похожие слова ('scared', 'worried' vs 'lonely', 'hopeless').
- anger <-> sadness: разочарование может выражаться словами обеих эмоций.
- love <-> joy: положительные эмоции часто смешиваются.
Ошибки с высокой уверенностью указывают на то, что граница между смежными эмоциями в тексте действительно размыта.

In [29]:
# Извлекаем историю логов из trainer
log_history = trainer.state.log_history

train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log_history if "eval_loss" in x]

if train_logs and eval_logs:
    train_steps  = [x["step"] for x in train_logs]
    train_losses = [x["loss"] for x in train_logs]
    eval_epochs  = [x["epoch"] for x in eval_logs]
    eval_losses  = [x["eval_loss"] for x in eval_logs]
    eval_accs    = [x["eval_accuracy"] for x in eval_logs]
    eval_f1s     = [x["eval_f1_macro"] for x in eval_logs]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(train_steps, train_losses, label="Train Loss")
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss"); axes[0].legend()

    axes[1].plot(eval_epochs, eval_losses, marker="o", label="Val Loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].set_title("Validation Loss"); axes[1].legend()

    axes[2].plot(eval_epochs, eval_accs, marker="o", label="Val Accuracy")
    axes[2].plot(eval_epochs, eval_f1s,  marker="s", label="Val F1 macro")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Score")
    axes[2].set_title("Validation Metrics"); axes[2].legend()

    plt.tight_layout()
    curves_path = os.path.join(ARTIFACTS_DIR, "training_curves.png")
    plt.savefig(curves_path, dpi=150)
    plt.show()
    print(f"Кривые обучения сохранены: {curves_path}")
else:
    print("Лог обучения пуст — кривые не построены.")

Кривые обучения сохранены: artifacts\training_curves.png


C:\Users\Kirill\AppData\Local\Temp\ipykernel_14380\1209838081.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
